# 04 - Constrained IK With QPs And `qpsol`

This notebook solves a position inverse-kinematics problem for a UR5 robot. Each iteration solves a QP with `ca.qpsol`.

The desired end-effector position is tracked through the cost. Joint limits are enforced as hard bounds on the step `dq`.

In [ ]:
# Colab setup.
# These tutorials assume a fresh Google Colab runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "casadi",
    "pin",
    "robot_descriptions",
    "numpy",
    "matplotlib",

])


If the next cell gives an error restart the session to load the libraries (ctrl + m + .) or click runtime -> restart session. Then rerun the second and third codeblocks (do not rerun the first block!).

In [ ]:
import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin

from robot_descriptions.loaders.pinocchio import load_robot_description

qp_opts = {"printLevel": "none", "print_time": False}

## Load UR5 And Choose A Feasible Target

We create a target by moving the robot to a known feasible configuration, then asking IK to recover only the end-effector position from the neutral configuration.

In [ ]:
robot = load_robot_description("ur5_description")
model = robot.model
data = model.createData()

tool_frame_id = model.getFrameId("tool0")
assert tool_frame_id < len(model.frames), "Could not find frame tool0"

q_min_raw = model.lowerPositionLimit.copy()
q_max_raw = model.upperPositionLimit.copy()
q_min = np.where(np.isfinite(q_min_raw), q_min_raw, -2.0 * np.pi)
q_max = np.where(np.isfinite(q_max_raw), q_max_raw, 2.0 * np.pi)

q0 = pin.neutral(model)
q0 = np.clip(q0, q_min + 1e-3, q_max - 1e-3)

q_target = q0 + np.array([0.4, -0.7, 0.5, -0.4, 0.3, 0.2])
q_target = np.clip(q_target, q_min + 0.05, q_max - 0.05)

pin.framesForwardKinematics(model, data, q_target)
desired_pose = data.oMf[tool_frame_id].copy()
desired_position = desired_pose.translation.copy()

pin.framesForwardKinematics(model, data, q0)
start_position = data.oMf[tool_frame_id].translation.copy()

print("nq:", model.nq, "nv:", model.nv)
print("start position:", start_position)
print("desired position:", desired_position)

## QP IK Formulation

At iteration `k`, Pinocchio gives the current end-effector position and translational Jacobian:

$$
p_k = p(q_k), \qquad J_k = \frac{\partial p}{\partial q}(q_k).
$$

The desired position comes from the translation part of the desired pose:

$$
p_{\mathrm{des}} = \mathrm{translation}(T_{\mathrm{des}}).
$$

Define the position error

$$
e_k = p_{\mathrm{des}} - p_k.
$$

The linearized position update is

$$
p(q_k + \Delta q) \approx p_k + J_k \Delta q.
$$

Each IK step solves

$$
\begin{aligned}
\min_{\Delta q \in \mathbb{R}^{n_q}} \quad
& \frac{1}{2}\|J_k \Delta q - e_k\|_2^2
+ \frac{\lambda}{2}\|\Delta q\|_2^2 \\
\text{s.t.} \quad
& q_{\min} - q_k \le \Delta q \le q_{\max} - q_k, \\
& -\Delta q_{\max}^{\mathrm{step}} \le \Delta q \le
\Delta q_{\max}^{\mathrm{step}}.
\end{aligned}
$$

Then the configuration is updated with Pinocchio integration:

$$
q_{k+1} = \mathrm{integrate}(q_k, \Delta q^*).
$$

## Build One Parametric QP Solver

At each iteration, the numeric Pinocchio Jacobian and position error are passed as parameters.

In [ ]:
nq = model.nq
position_dim = 3

dq = ca.SX.sym("dq", nq)
J_param = ca.SX.sym("J", position_dim, nq)
e_param = ca.SX.sym("e", position_dim)
p = ca.vertcat(ca.reshape(J_param, position_dim * nq, 1), e_param)

damping = 1e-6
objective = 0.5 * ca.sumsqr(J_param @ dq - e_param) + 0.5 * damping * ca.sumsqr(dq)

qp = {"x": dq, "p": p, "f": objective}
ik_step_solver = ca.qpsol("ik_step_solver", "qpoases", qp, qp_opts)

print(ik_step_solver)

## Iterative QP IK Loop

The position linearization is

`p(q + dq) ~= p(q) + J(q) dq`.

The QP minimizes `||J(q)dq - (p_desired - p(q))||^2` while enforcing `q_min <= q + dq <= q_max`.

In [ ]:
def frame_position(model, data, q, frame_id):
    pin.framesForwardKinematics(model, data, q)
    return data.oMf[frame_id].translation.copy()

q = q0.copy()
max_iterations = 80
tolerance = 1e-4
step_limit = 0.25

q_history = [q.copy()]
position_history = [frame_position(model, data, q, tool_frame_id)]
error_history = []

for iteration in range(max_iterations):
    pin.forwardKinematics(model, data, q)
    pin.updateFramePlacements(model, data)
    pin.computeJointJacobians(model, data, q)

    current_position = data.oMf[tool_frame_id].translation.copy()
    error = desired_position - current_position
    error_norm = np.linalg.norm(error)
    error_history.append(error_norm)

    if error_norm < tolerance:
        print(f"Converged in {iteration} iterations")
        break

    J6 = pin.computeFrameJacobian(
        model,
        data,
        q,
        tool_frame_id,
        pin.ReferenceFrame.LOCAL_WORLD_ALIGNED,
    )
    J_position = J6[:3, :]

    p_num = ca.vertcat(
        ca.reshape(ca.DM(J_position), position_dim * nq, 1),
        ca.DM(error),
    )

    lbx = np.maximum(q_min - q, -step_limit * np.ones(nq))
    ubx = np.minimum(q_max - q, step_limit * np.ones(nq))

    sol = ik_step_solver(
        x0=np.zeros(nq),
        p=p_num,
        lbx=lbx,
        ubx=ubx,
    )

    dq_solution = np.array(sol["x"]).reshape(-1)
    q = pin.integrate(model, q, dq_solution)
    q = np.clip(q, q_min, q_max)

    q_history.append(q.copy())
    position_history.append(frame_position(model, data, q, tool_frame_id))
else:
    print("Reached max_iterations without meeting tolerance")

q_history = np.asarray(q_history)
position_history = np.asarray(position_history)
final_error = np.linalg.norm(desired_position - position_history[-1])

print("final q:", q)
print("within joint limits:", bool(np.all(q >= q_min - 1e-8) and np.all(q <= q_max + 1e-8)))
print("final position:", position_history[-1])
print("desired position:", desired_position)
print("final position error norm:", final_error)

## Plots

The final plot shows convergence, joint trajectories, and the end-effector path.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(error_history, marker="o")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("position error norm")
axes[0].grid(True)

for joint_id in range(model.nq):
    axes[1].plot(q_history[:, joint_id], label=f"q{joint_id}")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("joint position")
axes[1].grid(True)
axes[1].legend(ncol=2, fontsize=8)

ax3 = fig.add_subplot(1, 3, 3, projection="3d")
ax3.plot(position_history[:, 0], position_history[:, 1], position_history[:, 2], marker="o", label="QP IK path")
ax3.scatter(*start_position, label="start", s=60)
ax3.scatter(*desired_position, label="desired", s=60)
ax3.scatter(*position_history[-1], label="final", s=60)
ax3.set_xlabel("x")
ax3.set_ylabel("y")
ax3.set_zlabel("z")
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Exercise

1. Change the target configuration and observe how the QP IK behaves.
2. Reduce `step_limit` and compare the convergence speed.
3. Extension: add orientation tracking by using all 6 rows of the frame Jacobian and a 6D pose error.